# 01 – Train Models: Step-by-Step

This notebook walks through the **fairness-auditbench** training pipeline:
1. Load the ACS Public Coverage dataset (Folktables)
2. Train a Logistic Regression
3. Train an FT-Transformer
4. Inspect saved artefacts

> **Tip:** Run this on the cluster after executing `scripts/local_scripts/cluster_install_gpu.sh`.

In [1]:
import sys, os

# Ensure src/ is importable (if not using editable install)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import fairness_auditbench
print(f"fairness_auditbench v{fairness_auditbench.__version__}")

fairness_auditbench v0.1.0


## 1. Configuration

In [2]:
from fairness_auditbench.config import TrainConfig

config = TrainConfig(
    dataset="acs_public_coverage",
    states=["CA"],
    year=2018,
    seed=0,
    fast_dev_run=False,
    fast_dev_n=500,
    data_dir=os.path.join(repo_root, "data"),
    output_dir=os.path.join(repo_root, "results"),
)
print(config)

TrainConfig(dataset='acs_public_coverage', states=['CA'], year=2018, horizon='1-Year', survey='person', sensitive_col=None, model='logreg', seed=0, fast_dev_run=False, fast_dev_n=500, data_dir='/storage/coda1/p-vzikas3/0/ywei368/Yu-Project/SyncDP/fairness-auditbench/data', output_dir='/storage/coda1/p-vzikas3/0/ywei368/Yu-Project/SyncDP/fairness-auditbench/results', max_epochs=50, batch_size=256, lr=0.0001, patience=5, logreg_max_iter=1000)


## 2. Load Dataset

In [3]:
from fairness_auditbench.datasets import get_dataset

ds = get_dataset(
    config.dataset,
    states=config.states,
    year=config.year,
    data_dir=config.data_dir,
    fast_dev_run=config.fast_dev_run,
    fast_dev_n=config.fast_dev_n,
)
train_df, val_df, test_df, spec = ds.get_splits(seed=config.seed)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Label: {spec.label_col}, Sensitive: {spec.sensitive_col}")
print(f"Categorical cols ({len(spec.categorical_cols)}): {spec.categorical_cols[:5]}")
print(f"Numerical cols ({len(spec.numerical_cols)}): {spec.numerical_cols[:5]}")
train_df.head()

Train: 96987, Val: 13855, Test: 27712
Label: PUBCOV, Sensitive: RAC1P
Categorical cols (15): ['MAR', 'SEX', 'DIS', 'ESP', 'CIT']
Numerical cols (3): ['AGEP', 'SCHL', 'PINCP']


,AGEP,SCHL,MAR,SEX,DIS,ESP,CIT,MIG,MIL,ANC,NATIVITY,DEAR,DEYE,DREM,PINCP,ESR,ST,FER,RAC1P,PUBCOV
1931,56.0,18.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,13450.0,6.0,6.0,0.0,1.0,False
52671,24.0,21.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,2.0,1.0,2.0,2.0,2.0,360.0,1.0,6.0,0.0,8.0,False
10853,28.0,16.0,5.0,1.0,2.0,0.0,5.0,1.0,4.0,4.0,2.0,2.0,2.0,2.0,0.0,6.0,6.0,0.0,8.0,False
138373,18.0,14.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,0.0,6.0,6.0,0.0,1.0,False
15397,55.0,16.0,1.0,2.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,0.0,6.0,6.0,0.0,1.0,True


## 3. Train Logistic Regression

In [4]:
from fairness_auditbench.models.logreg import LogisticRegressionModel

lr_model = LogisticRegressionModel()
lr_metrics = lr_model.train_model(train_df, val_df, spec, config)
print("LogReg metrics:", lr_metrics)

lr_dir = os.path.join(config.output_dir, "models", config.dataset, "logreg", f"seed={config.seed}")
lr_model.save(lr_dir)
print(f"Saved to {lr_dir}")

/tmp/python-venv/fairness-auditbench_venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LogReg metrics: {'accuracy': 0.694478527607362, 'auroc': 0.7016821147505976}
Saved to /storage/coda1/p-vzikas3/0/ywei368/Yu-Project/SyncDP/fairness-auditbench/results/models/acs_public_coverage/logreg/seed=0


## 4. Train FT-Transformer

In [5]:
from fairness_auditbench.models.ft_transformer import FTTransformerModel

config.max_epochs = 50
config.model = "ft_transformer"

ft_model = FTTransformerModel()
ft_metrics = ft_model.train_model(train_df, val_df, spec, config)
print("FT-Transformer metrics:", ft_metrics)

ft_dir = os.path.join(config.output_dir, "models", config.dataset, "ft_transformer", f"seed={config.seed}")
ft_model.save(ft_dir)
print(f"Saved to {ft_dir}")

/tmp/python-venv/fairness-auditbench_venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


FT-Transformer metrics: {'accuracy': 0.7137495488993143, 'auroc': 0.7549002090113852}
Saved to /storage/coda1/p-vzikas3/0/ywei368/Yu-Project/SyncDP/fairness-auditbench/results/models/acs_public_coverage/ft_transformer/seed=0


## 5. Inspect Saved Artefacts

In [6]:
import json

for model_name in ["logreg", "ft_transformer"]:
    mdir = os.path.join(config.output_dir, "models", config.dataset, model_name, f"seed={config.seed}")
    metrics_file = os.path.join(mdir, "metrics.json")
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f"{model_name}: {m}")
        print(f"  Files: {os.listdir(mdir)}")
    else:
        print(f"{model_name}: not yet trained")

logreg: {'accuracy': 0.694478527607362, 'auroc': 0.7016821147505976}
  Files: ['pipeline.joblib', 'metrics.json']
ft_transformer: {'accuracy': 0.7137495488993143, 'auroc': 0.7549002090113852}
  Files: ['model.pt', 'metrics.json', 'hparams.json', 'preprocessor.joblib']
